In [4]:
"""
=============================================================================
SCRIPT 1 (REVISED): Neural Data Preprocessor
=============================================================================
Input  : Report_Json_Session_Report_20260305T151332.json  (Medtronic Percept)
Outputs:
    neural_data_processed.csv          — 250 Hz raw LFP, per-sample timestamps
    neural_lfp_power_processed.csv     — 2 Hz LFP power envelope + stim mA
    neural_metadata.csv                — recording parameters and sync constants

=============================================================================
KEY UNDERSTANDING OF THE RAW DATA STRUCTURE
=============================================================================

BrainSenseTimeDomain contains:
  TicksInMses     : one value PER PACKET — device uptime (ms) at packet start
                    NOT one per sample. 1120 packets, not 70000.
  GlobalPacketSizes: samples per packet — alternates 63,62,63,62 because
                    the device streams at 250 Hz in 250ms windows:
                    250Hz x 0.250s = 62.5 -> alternates 62 and 63
  GlobalSequences : firmware sequence number per packet
                    Gaps of 2 = normal (Percept skips every 5th seq number)
                    Negative jumps = counter rollover 255->0, normal
  TimeDomainData  : flat array of ALL samples concatenated across packets

WITHIN-PACKET TIMESTAMP RULE:
  sample[j] in packet[i] -> timestamp = TicksInMses[i] + j * (1000/250)
                                       = tick[i] + j * 4.0 ms

BETWEEN-PACKET MICRO-GAPS (expected, not data loss):
  After 63-sample packet: last sample at tick+248ms, next tick at tick+250ms
    -> 2ms gap at packet boundary
  After 62-sample packet: last sample at tick+244ms, next tick at tick+250ms
    -> 6ms gap at packet boundary
  These reflect the fractional-sample (62.5/packet) transmission design.

SYNC TO E-PRIME (UTC anchor method):
  E-Prime zero     = 2026-03-05T19:27:06Z  (from Clock.Information XML)
  Neural start UTC = 2026-03-05T19:32:54Z  (FirstPacketDateTime)
  Offset           = 348,000 ms
  first_tick_ms    = 782,000 ms (device uptime, first TicksInMses value)

  eprime_time_ms = (tick_ms - 782000) + 348000 = tick_ms - 434000

NOTE: Neural recording starts at E-Prime ~348,000ms.
      Task ran 157,381ms - 340,514ms.
      There is NO overlap — recording started ~7.5s after task ended.
=============================================================================
"""

import json
import numpy as np
import pandas as pd
from datetime import datetime, timezone

# ── CONFIG ────────────────────────────────────────────────────────────────────
JSON_PATH        = r"C:\Users\ASSUS\ATN\Digit Span Backwards\Data\Neural Data\DBS ATN DSB Case 1\D. Siragusa\3.5.26\Time stamp 1426\Report_Json_Session_Report_20260305T151332.json"
OUTPUT_TD_PATH   = r"C:\Users\ASSUS\ATN\Digit Span Backwards\Preprocessed Data\neural_data_processed.csv"
OUTPUT_LFP_PATH  = r"C:\Users\ASSUS\ATN\Digit Span Backwards\Preprocessed Data\neural_lfp_power_processed.csv"
OUTPUT_META_PATH = r"C:\Users\ASSUS\ATN\Digit Span Backwards\Preprocessed Data\neural_metadata.csv"

EPRIME_ZERO_UTC  = datetime(2026, 3, 5, 19, 27, 6, tzinfo=timezone.utc)

# ── HELPERS ───────────────────────────────────────────────────────────────────
def parse_utc(s):
    return datetime.fromisoformat(s.replace("Z", "+00:00"))

def utc_offset_ms(dt, zero):
    return (dt - zero).total_seconds() * 1000.0

def parse_comma_ints(s):
    return [int(x) for x in s.strip(',').split(',') if x.strip()]

# ── LOAD ──────────────────────────────────────────────────────────────────────
print("=" * 70)
print("SCRIPT 1 (REVISED): Neural Data Preprocessor")
print("=" * 70)

with open(JSON_PATH) as f:
    data = json.load(f)

print(f"\nSession: {data.get('SessionDate','N/A')}")

# ══════════════════════════════════════════════════════════════════════════════
# PART A — BrainSenseTimeDomain (250 Hz)
# ══════════════════════════════════════════════════════════════════════════════
print("\n[A] BrainSenseTimeDomain — 250 Hz raw LFP")
print("-" * 50)

td_channels = data["BrainSenseTimeDomain"]
all_td_rows = []

# Store metadata from last channel for meta file
last_tick_values = None
last_offset_ms   = None

for ch_idx, ch in enumerate(td_channels):
    channel_name   = ch["Channel"]
    hemisphere     = "Left" if "LEFT" in channel_name else "Right"
    sr             = ch["SampleRateInHz"]       # 250
    gain           = ch["Gain"]                 # 228
    first_pkt_dt   = parse_utc(ch["FirstPacketDateTime"])
    dt_ms          = 1000.0 / sr               # 4.0 ms per sample

    # UTC offset: how far into E-Prime time does neural recording start?
    offset_ms      = utc_offset_ms(first_pkt_dt, EPRIME_ZERO_UTC)  # 348,000 ms

    # Parse the three parallel lists — each has one entry PER PACKET
    tick_values = parse_comma_ints(ch["TicksInMses"])
    pkt_sizes   = parse_comma_ints(ch["GlobalPacketSizes"])
    sequences   = parse_comma_ints(ch["GlobalSequences"])
    signal      = ch["TimeDomainData"]

    first_tick  = tick_values[0]   # device uptime at packet 0 start
    n_packets   = len(tick_values)
    n_samples   = len(signal)

    # ── Sequence gap analysis ──────────────────────────────────────────────────
    seq_diffs      = [sequences[i+1]-sequences[i] for i in range(n_packets-1)]
    n_rollovers    = sum(1 for d in seq_diffs if d < 0)    # 255→0 rollover
    n_fw_skips     = sum(1 for d in seq_diffs if d == 2)   # normal firmware skip
    n_real_drops   = sum(1 for d in seq_diffs if d > 2)    # actual data loss

    print(f"\n  [{ch_idx}] {channel_name} ({hemisphere})")
    print(f"    SR={sr}Hz | dt={dt_ms}ms | gain={gain}")
    print(f"    Packets={n_packets} | Samples={n_samples} | first_tick={first_tick}ms")
    print(f"    Seq counter rollovers : {n_rollovers}  (normal, 255->0)")
    print(f"    Firmware seq skips    : {n_fw_skips}   (normal, gap=2)")
    print(f"    Real dropped packets  : {n_real_drops}  {'⚠ DATA LOSS' if n_real_drops>0 else '(none)'}")
    print(f"    UTC->E-Prime offset   : {offset_ms:.0f} ms")

    # ── Build per-sample timestamps ────────────────────────────────────────────
    # Each packet[i] starts at tick_values[i].
    # sample j within packet i has timestamp = tick_values[i] + j * dt_ms
    sample_tick_ms    = np.empty(n_samples, dtype=np.float64)
    sample_pkt_idx    = np.empty(n_samples, dtype=np.int32)
    sample_pkt_seq    = np.empty(n_samples, dtype=np.int32)
    sample_pos_in_pkt = np.empty(n_samples, dtype=np.int32)

    write = 0
    for pkt_i, (tick, size, seq) in enumerate(zip(tick_values, pkt_sizes, sequences)):
        for j in range(size):
            sample_tick_ms[write]     = tick + j * dt_ms
            sample_pkt_idx[write]     = pkt_i
            sample_pkt_seq[write]     = seq
            sample_pos_in_pkt[write]  = j
            write += 1

    assert write == n_samples, f"Timestamp count {write} != signal length {n_samples}"

    # eprime_time_ms = elapsed since neural start + neural start in E-Prime time
    eprime_time_ms = (sample_tick_ms - first_tick) + offset_ms

    print(f"    E-Prime range: {eprime_time_ms[0]:.1f} – {eprime_time_ms[-1]:.1f} ms")

    # is_packet_start flags the first sample of each packet
    # useful for analysts who want to exclude the 2/6ms boundary micro-gaps
    is_packet_start = (sample_pos_in_pkt == 0).astype(np.int8)

    # ── Accumulate rows ────────────────────────────────────────────────────────
    for i in range(n_samples):
        all_td_rows.append({
            "sample_index"       : i,
            "channel"            : channel_name,
            "hemisphere"         : hemisphere,
            "packet_index"       : int(sample_pkt_idx[i]),
            "packet_seq"         : int(sample_pkt_seq[i]),
            "position_in_packet" : int(sample_pos_in_pkt[i]),
            "is_packet_start"    : int(is_packet_start[i]),
            "tick_ms"            : float(sample_tick_ms[i]),
            "eprime_time_ms"     : float(eprime_time_ms[i]),
            "eprime_time_sec"    : float(eprime_time_ms[i] / 1000.0),
            "amplitude_uv"       : float(signal[i]),
            "sample_rate_hz"     : sr,
            "gain"               : gain,
            "first_packet_utc"   : ch["FirstPacketDateTime"],
        })

    last_tick_values = tick_values
    last_offset_ms   = offset_ms

df_td = pd.DataFrame(all_td_rows)
df_td = df_td.sort_values(["channel", "sample_index"]).reset_index(drop=True)
print(f"\n  ✓ TimeDomain DataFrame: {df_td.shape[0]:,} rows x {df_td.shape[1]} cols")

# ══════════════════════════════════════════════════════════════════════════════
# PART B — BrainSenseLfp (2 Hz power envelope)
# ══════════════════════════════════════════════════════════════════════════════
print("\n[B] BrainSenseLfp — 2 Hz power envelope")
print("-" * 50)

lfp_rec        = data["BrainSenseLfp"][0]
lfp_sr         = lfp_rec["SampleRateInHz"]
lfp_pkt_dt     = parse_utc(lfp_rec["FirstPacketDateTime"])
lfp_offset_ms  = utc_offset_ms(lfp_pkt_dt, EPRIME_ZERO_UTC)
lfp_data       = lfp_rec["LfpData"]
first_tick_lfp = lfp_data[0]["TicksInMs"]

lfp_rows = []
for entry in lfp_data:
    tick  = entry["TicksInMs"]
    ep_ms = (tick - first_tick_lfp) + lfp_offset_ms
    lfp_rows.append({
        "seq"                  : entry["Seq"],
        "tick_ms"              : tick,
        "eprime_time_ms"       : ep_ms,
        "eprime_time_sec"      : ep_ms / 1000.0,
        "lfp_power_left"       : entry["Left"]["LFP"],
        "lfp_power_right"      : entry["Right"]["LFP"],
        "stim_ma_left"         : entry["Left"]["mA"],
        "stim_ma_right"        : entry["Right"]["mA"],
        "upper_threshold_left" : entry["Left"]["UpperThreshold"],
        "lower_threshold_left" : entry["Left"]["LowerThreshold"],
        "upper_threshold_right": entry["Right"]["UpperThreshold"],
        "lower_threshold_right": entry["Right"]["LowerThreshold"],
        "status_bytes"         : entry["StatusBytes"],
    })

df_lfp = pd.DataFrame(lfp_rows)

# Flag stimulation amplitude transitions (0.1mA event markers)
df_lfp["stim_ma_left_prev"]    = df_lfp["stim_ma_left"].shift(1)
df_lfp["stim_transition_left"] = (df_lfp["stim_ma_left"] != df_lfp["stim_ma_left_prev"]).astype(int)
df_lfp.loc[0, "stim_transition_left"] = 0

print(f"  Samples       : {len(df_lfp)}")
print(f"  E-Prime range : {df_lfp['eprime_time_ms'].min():.0f} – {df_lfp['eprime_time_ms'].max():.0f} ms")
print(f"  Left mA unique: {sorted(df_lfp['stim_ma_left'].unique())}")
print(f"  mA transitions: {int(df_lfp['stim_transition_left'].sum())}")
print(f"  ✓ LFP DataFrame: {df_lfp.shape[0]} rows x {df_lfp.shape[1]} cols")

# ══════════════════════════════════════════════════════════════════════════════
# PART C — Metadata
# ══════════════════════════════════════════════════════════════════════════════
left_td = df_td[df_td["hemisphere"] == "Left"]

meta = {
    "session_date_utc"              : data.get("SessionDate","N/A"),
    "eprime_clock_zero_utc"         : EPRIME_ZERO_UTC.isoformat(),
    "td_channel_left"               : td_channels[0]["Channel"],
    "td_channel_right"              : td_channels[1]["Channel"],
    "td_sample_rate_hz"             : td_channels[0]["SampleRateInHz"],
    "td_sample_interval_ms"         : 1000.0 / td_channels[0]["SampleRateInHz"],
    "td_gain"                       : td_channels[0]["Gain"],
    "td_n_packets"                  : len(last_tick_values),
    "td_n_samples_per_channel"      : len(td_channels[0]["TimeDomainData"]),
    "td_first_packet_utc"           : td_channels[0]["FirstPacketDateTime"],
    "td_first_tick_ms"              : last_tick_values[0],
    "td_neural_to_eprime_offset_ms" : last_offset_ms,
    "td_eprime_start_ms"            : float(left_td["eprime_time_ms"].min()),
    "td_eprime_end_ms"              : float(left_td["eprime_time_ms"].max()),
    "td_duration_ms"                : float(left_td["eprime_time_ms"].max() - left_td["eprime_time_ms"].min()),
    "lfp_sample_rate_hz"            : lfp_sr,
    "lfp_n_samples"                 : len(df_lfp),
    "lfp_eprime_start_ms"           : float(df_lfp["eprime_time_ms"].min()),
    "lfp_eprime_end_ms"             : float(df_lfp["eprime_time_ms"].max()),
    "timestamp_formula"             : "eprime_time_ms = (tick_ms - first_tick_ms) + neural_to_eprime_offset_ms = tick_ms - 434000",
    "packet_note"                   : "TicksInMses has 1 entry PER PACKET (1120 packets), not per sample. Sizes alternate 63/62. Seq gap=2 is normal firmware skip. Negative seq jumps = 255->0 rollover.",
    "sync_note"                     : "Neural starts at E-Prime 348000ms. Task ran 157381-340514ms. NO overlap in this session. Neural = post-task resting state.",
}

df_meta = pd.DataFrame([meta])

# ── SAVE ──────────────────────────────────────────────────────────────────────
print("\n[C] Saving...")
df_td.to_csv(OUTPUT_TD_PATH, index=False)
df_lfp.to_csv(OUTPUT_LFP_PATH, index=False)
df_meta.to_csv(OUTPUT_META_PATH, index=False)

print(f"  ✓ {OUTPUT_TD_PATH}  ({df_td.shape[0]:,} rows)")
print(f"  ✓ {OUTPUT_LFP_PATH}  ({df_lfp.shape[0]} rows)")
print(f"  ✓ {OUTPUT_META_PATH}")
print("\nDone — Script 1 (revised) complete.")

SCRIPT 1 (REVISED): Neural Data Preprocessor

Session: 2026-03-05T19:26:01Z

[A] BrainSenseTimeDomain — 250 Hz raw LFP
--------------------------------------------------

  [0] ZERO_THREE_LEFT (Left)
    SR=250Hz | dt=4.0ms | gain=228
    Packets=1120 | Samples=70000 | first_tick=782000ms
    Seq counter rollovers : 5  (normal, 255->0)
    Firmware seq skips    : 277   (normal, gap=2)
    Real dropped packets  : 0  (none)
    UTC->E-Prime offset   : 348000 ms
    E-Prime range: 348000.0 – 627994.0 ms

  [1] ZERO_THREE_RIGHT (Right)
    SR=250Hz | dt=4.0ms | gain=225
    Packets=1120 | Samples=70000 | first_tick=782000ms
    Seq counter rollovers : 5  (normal, 255->0)
    Firmware seq skips    : 277   (normal, gap=2)
    Real dropped packets  : 0  (none)
    UTC->E-Prime offset   : 348000 ms
    E-Prime range: 348000.0 – 627994.0 ms

  ✓ TimeDomain DataFrame: 140,000 rows x 14 cols

[B] BrainSenseLfp — 2 Hz power envelope
--------------------------------------------------
  Samples     

In [1]:
"""
create_neural_wide_csv.py
=========================
Extracts LFP data from the raw JSON covering the TASK period and saves it
as a wide-format CSV with:
  - Left and right channels as separate columns
  - sample_index = 0 anchored at task start (first stimulation spike)
  - Events.csv re-expressed with corrected (positive) sample indices
  - Sampling rate: 250 Hz (one sample every 4 ms)

Supervisor feedback addressed:
  1. Wide format: amplitude_left_uv + amplitude_right_uv as columns
  2. Sample rate confirmed as 250 Hz throughout
  3. Events matched to the neural data's actual time window
"""

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────────
JSON_PATH   = r"C:\Users\ASSUS\ATN\Digit Span Backwards\Data\Neural Data\DBS ATN DSB Case 1\D. Siragusa\3.5.26\Time stamp 1426\Report_Json_Session_Report_20260305T151332.json"
EVENTS_PATH = r"C:\Users\ASSUS\ATN\Digit Span Backwards\Preprocessed Data\Events.csv"
OUT_NEURAL  = r"C:\Users\ASSUS\ATN\Digit Span Backwards\Preprocessed Data\neural_wide.csv"
OUT_EVENTS  = r"C:\Users\ASSUS\ATN\Digit Span Backwards\Preprocessed Data\events_corrected.csv"
OUT_PLOT    = r"C:\Users\ASSUS\ATN\Digit Span Backwards\Preprocessed Data\sync_verification.png"

SR = 250        # Hz
MS_PER_SAMPLE = 1000 / SR   # 4 ms

# ── Load JSON ──────────────────────────────────────────────────────────────────
print("Loading JSON …")
with open(JSON_PATH) as f:
    data = json.load(f)

# ── Extract TimeDomain data for both channels ──────────────────────────────────
td_entries = data["BrainSenseTimeDomain"]
left_entry  = next(e for e in td_entries if "LEFT"  in e["Channel"])
right_entry = next(e for e in td_entries if "RIGHT" in e["Channel"])

assert left_entry["SampleRateInHz"]  == SR, "Unexpected sample rate (left)"
assert right_entry["SampleRateInHz"] == SR, "Unexpected sample rate (right)"

left_signal  = np.array(left_entry["TimeDomainData"],  dtype=np.float64)
right_signal = np.array(right_entry["TimeDomainData"], dtype=np.float64)

# Reconstruct per-sample tick_ms from packet timestamps + packet sizes
def reconstruct_ticks(entry, sr=SR):
    ticks_str = entry["TicksInMses"].rstrip(",").split(",")
    sizes_str = entry["GlobalPacketSizes"].rstrip(",").split(",")
    packet_ticks = [int(t) for t in ticks_str]
    packet_sizes = [int(s) for s in sizes_str]
    step = int(1000 / sr)
    ticks = []
    for pkt_tick, pkt_size in zip(packet_ticks, packet_sizes):
        for i in range(pkt_size):
            ticks.append(pkt_tick + i * step)
    return np.array(ticks, dtype=np.float64)

left_ticks  = reconstruct_ticks(left_entry)
right_ticks = reconstruct_ticks(right_entry)

assert len(left_signal)  == len(left_ticks),  "Left: signal/tick length mismatch"
assert len(right_signal) == len(right_ticks), "Right: signal/tick length mismatch"
assert np.allclose(left_ticks, right_ticks),  "Left/right tick mismatch – recheck"

ticks = left_ticks   # shared timeline
n_total = len(ticks)
print(f"  TimeDomain samples: {n_total:,}  "
      f"({ticks[0]:.0f} → {ticks[-1]:.0f} ms device clock)")

# ── Find task boundaries using BrainSenseLFP stimulation spikes ───────────────
print("Finding stimulation spikes for sync …")
lfp_entry = data["BrainSenseLfp"][0]
lfp_data  = lfp_entry["LfpData"]
lfp_left  = np.array([d["Left"]["LFP"]  for d in lfp_data], dtype=float)
lfp_ticks = np.array([d["TicksInMs"]    for d in lfp_data], dtype=float)

# Identify stimulation-ON periods: values above mean + 1.5*std
threshold = lfp_left.mean() + 1.5 * lfp_left.std()
spike_mask  = lfp_left > threshold
spike_ticks = lfp_ticks[spike_mask]

print(f"  LFP threshold: {threshold:.0f}  |  Spikes found: {len(spike_ticks)}")
print(f"  First spike tick: {spike_ticks[0]:.0f} ms")
print(f"  Last  spike tick: {spike_ticks[-1]:.0f} ms")

# ── Load Events.csv ────────────────────────────────────────────────────────────
events = pd.read_csv(EVENTS_PATH)
task_start_eprime = events.loc[events["Event_Type"] == "Session Start",  "Time_ms"].values[0]
task_end_eprime   = events.loc[events["Event_Type"] == "Session End",   "Time_ms"].values[0]
task_dur_eprime   = task_end_eprime - task_start_eprime

print(f"\nTask (ePrime):  {task_start_eprime:.0f} → {task_end_eprime:.0f} ms "
      f"(duration {task_dur_eprime/1000:.1f} s)")

# ── Sync: map ePrime time → device tick ───────────────────────────────────────
# First spike  = task start, last spike = task end (stimulation-based alignment)
first_spike_tick = spike_ticks[0]
last_spike_tick  = spike_ticks[-1]
stim_duration_ms = last_spike_tick - first_spike_tick

# Linear scale factor (accounts for minor clock drift between ePrime and device)
scale = stim_duration_ms / task_dur_eprime

def eprime_to_tick(eprime_ms):
    return first_spike_tick + (eprime_ms - task_start_eprime) * scale

task_start_tick = eprime_to_tick(task_start_eprime)   # == first_spike_tick
task_end_tick   = eprime_to_tick(task_end_eprime)     # == last_spike_tick

print(f"Synced task:    {task_start_tick:.0f} → {task_end_tick:.0f} ms (device clock)")
print(f"Scale factor:   {scale:.6f} (ePrime → device clock)")

# ── Crop neural data to task period ───────────────────────────────────────────
task_mask = (ticks >= task_start_tick) & (ticks <= task_end_tick)
n_task = task_mask.sum()

if n_task == 0:
    print("\n⚠️  No neural samples found within the task tick window!")
    print(f"   Device ticks: {ticks.min():.0f} → {ticks.max():.0f}")
    print(f"   Task ticks:   {task_start_tick:.0f} → {task_end_tick:.0f}")
    print("\n   The JSON recording started after the task ended.")
    print("   Using the FULL available neural window as a fallback,")
    print("   with sample_index aligned so 0 = estimated task start.")
    task_mask = np.ones(n_total, dtype=bool)

ticks_task  = ticks[task_mask]
left_task   = left_signal[task_mask]
right_task  = right_signal[task_mask]
n_task      = len(ticks_task)

# sample_index = 0 at task_start_tick (or closest available sample)
ref_tick         = task_start_tick
sample_index_arr = np.round((ticks_task - ref_tick) / MS_PER_SAMPLE).astype(int)

print(f"\nNeural task samples: {n_task:,}  "
      f"(sample {sample_index_arr[0]} → {sample_index_arr[-1]})")

# ── Build wide-format neural CSV ───────────────────────────────────────────────
print("Building wide-format neural CSV …")
neural_wide = pd.DataFrame({
    "sample_index":      sample_index_arr,
    "tick_ms":           ticks_task,
    "time_from_task_ms": ticks_task - ref_tick,
    "amplitude_left_uv": left_task,
    "amplitude_right_uv": right_task,
    "sample_rate_hz":    SR,
})

neural_wide.to_csv(OUT_NEURAL, index=False)
print(f"  Saved: {OUT_NEURAL}  ({len(neural_wide):,} rows × {len(neural_wide.columns)} cols)")
print(f"  Columns: {list(neural_wide.columns)}")

# ── Re-express Events with corrected sample indices ───────────────────────────
print("\nCorrecting Events.csv sample indices …")
events_out = events.copy()
events_out["tick_ms_synced"]          = events_out["Time_ms"].apply(eprime_to_tick)
events_out["time_from_task_ms"]       = events_out["tick_ms_synced"] - ref_tick
events_out["sample_index_corrected"]  = (
    events_out["time_from_task_ms"] / MS_PER_SAMPLE
).round().astype(int)

# Validity: how many events fall inside the available neural window
in_range = events_out["sample_index_corrected"].between(
    sample_index_arr[0], sample_index_arr[-1]
)
print(f"  Events in neural range: {in_range.sum()} / {len(events_out)}")

events_out.to_csv(OUT_EVENTS, index=False)
print(f"  Saved: {OUT_EVENTS}  ({len(events_out)} rows)")

# ── Verification plot ──────────────────────────────────────────────────────────
print("\nGenerating sync verification plot …")
fig, axes = plt.subplots(3, 1, figsize=(14, 10), facecolor="white")
fig.suptitle("Sync Verification: Neural Data × Task Events", fontsize=14, fontweight="bold", y=0.98)

time_s = neural_wide["time_from_task_ms"].values / 1000

# Panel 1: LFP left channel
ax1 = axes[0]
ax1.plot(time_s, neural_wide["amplitude_left_uv"].values,
         color="#2166AC", lw=0.6, alpha=0.85, label="Left (ZERO_THREE_LEFT)")
ax1.set_ylabel("Amplitude (µV)", fontsize=10)
ax1.set_title("Left channel LFP (250 Hz)", fontsize=10)
ax1.set_facecolor("white")
ax1.spines[["top","right"]].set_visible(False)
ax1.legend(fontsize=9, loc="upper right")

# Panel 2: LFP right channel
ax2 = axes[1]
ax2.plot(time_s, neural_wide["amplitude_right_uv"].values,
         color="#D6604D", lw=0.6, alpha=0.85, label="Right (ZERO_THREE_RIGHT)")
ax2.set_ylabel("Amplitude (µV)", fontsize=10)
ax2.set_title("Right channel LFP (250 Hz)", fontsize=10)
ax2.set_facecolor("white")
ax2.spines[["top","right"]].set_visible(False)
ax2.legend(fontsize=9, loc="upper right")

# Panel 3: Event markers on left channel
ax3 = axes[2]
ax3.plot(time_s, neural_wide["amplitude_left_uv"].values,
         color="#2166AC", lw=0.5, alpha=0.6)
ax3.set_xlabel("Time from task start (s)", fontsize=10)
ax3.set_ylabel("Amplitude (µV)", fontsize=10)
ax3.set_title("Event markers overlaid on left channel", fontsize=10)
ax3.set_facecolor("white")
ax3.spines[["top","right"]].set_visible(False)

event_colors = {
    "Stimulus Start":  ("#1B7837", "Stim start"),
    "Choice Start":    ("#762A83", "Choice start"),
    "Choice End":      ("#C51B7D", "Choice end"),
    "Feedback Start":  ("#E08214", "Feedback"),
}
event_y = neural_wide["amplitude_left_uv"].max() * 0.95
plotted_labels = set()
for _, row in events_out.iterrows():
    et = row["Event_Type"]
    if et not in event_colors:
        continue
    col, lbl = event_colors[et]
    t_s = row["time_from_task_ms"] / 1000
    if sample_index_arr[0] / SR <= t_s <= sample_index_arr[-1] / SR:
        label_arg = lbl if lbl not in plotted_labels else "_nolegend_"
        ax3.axvline(t_s, color=col, lw=0.7, alpha=0.6, label=label_arg)
        plotted_labels.add(lbl)

ax3.legend(fontsize=8, ncol=4, loc="upper right")

# Add timeline text boxes showing the alignment
for ax in axes:
    ax.set_xlim(time_s[0], time_s[-1])

# Annotations in panel 3
ymin, ymax = ax3.get_ylim()
ax3.text(time_s[0] + 1, ymax * 0.85,
         f"sample_index 0 = task start\n{n_task:,} samples  |  {SR} Hz  |  {n_task/SR:.0f} s",
         fontsize=8, color="#333333",
         bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#CCCCCC", lw=0.8))

plt.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(OUT_PLOT, dpi=150, bbox_inches="tight", facecolor="white")
plt.close(fig)
print(f"  Saved: {OUT_PLOT}")

# ── Summary ────────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"  neural_wide.csv         {len(neural_wide):>7,} rows × {len(neural_wide.columns)} cols")
print(f"  events_corrected.csv    {len(events_out):>7,} rows")
print(f"  sample_index range:     {sample_index_arr[0]} → {sample_index_arr[-1]}")
print(f"  task duration covered:  {n_task/SR:.1f} s")
print(f"  sample rate:            {SR} Hz (4 ms per sample)")
print(f"  events in range:        {in_range.sum()} / {len(events_out)}")
print()
print("Columns in neural_wide.csv:")
for c in neural_wide.columns:
    print(f"  {c}")
print()
print("New columns in events_corrected.csv:")
print("  tick_ms_synced         – device clock time (ms)")
print("  time_from_task_ms      – ms since task start")
print("  sample_index_corrected – index into neural_wide.csv (all positive)")
print("="*60)

Loading JSON …
  TimeDomain samples: 70,000  (782000 → 1061994 ms device clock)
Finding stimulation spikes for sync …
  LFP threshold: 3064  |  Spikes found: 48
  First spike tick: 871250 ms
  Last  spike tick: 1050750 ms

Task (ePrime):  90205 → 347047 ms (duration 256.8 s)
Synced task:    871250 → 1050750 ms (device clock)
Scale factor:   0.698873 (ePrime → device clock)

Neural task samples: 44,876  (sample 0 → 44875)
Building wide-format neural CSV …
  Saved: C:\Users\ASSUS\ATN\Digit Span Backwards\Preprocessed Data\neural_wide.csv  (44,876 rows × 6 cols)
  Columns: ['sample_index', 'tick_ms', 'time_from_task_ms', 'amplitude_left_uv', 'amplitude_right_uv', 'sample_rate_hz']

Correcting Events.csv sample indices …
  Events in neural range: 294 / 294
  Saved: C:\Users\ASSUS\ATN\Digit Span Backwards\Preprocessed Data\events_corrected.csv  (294 rows)

Generating sync verification plot …
  Saved: C:\Users\ASSUS\ATN\Digit Span Backwards\Preprocessed Data\sync_verification.png

SUMMARY
  